# Lab 11 – Relevance Scoring & Rerankers for Trustworthy AI / EU AI Act

This notebook builds an advanced RAG pipeline that answers questions over two document types:
- **EU AI policy document** (PDF – regulation / guidelines)
- **Trustworthy AI podcast transcript** (text – conversational)

We measure how much retrieval quality improves when we add **metadata filtering** and **reranking** on top of a baseline similarity-search pipeline.

| Step | Required | What it does |
|------|----------|--------------|
| 1 | Mandatory | Load docs, chunk with metadata |
| 2 | Mandatory | Baseline Pinecone similarity search |
| 3 | Advanced | LLM-based relevance scoring |
| 4 | Advanced | Cross-encoder / Cohere reranker |
| 5 | Mandatory | Metadata filtering |
| 6 | Mandatory | Full LCEL RAG chain with reranking |
| 7 | Mandatory | Side-by-side evaluation table |

## Step 1 – Setup & Data Preparation

**Objective:** Load the two source documents and chunk them with rich metadata so that later steps can filter by source type.

In [160]:
import os
import glob
from dotenv import load_dotenv

load_dotenv()

# Verify keys are present
for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    assert os.environ.get(key), f"{key} is not set – check your .env file"
print("Environment OK")

Environment OK


In [161]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Load EU AI policy document ---
pdf_path = "data/eu_ai_act.pdf"
ai_act_loader = PyPDFLoader(pdf_path)
ai_act_docs = ai_act_loader.load()

for doc in ai_act_docs:
    doc.metadata["source_type"] = "regulation"
    doc.metadata["doc_name"] = "EU AI Policy"

print(f"Loaded {len(ai_act_docs)} pages from {pdf_path}")

Loaded 56 pages from data/eu_ai_act.pdf


In [162]:
# --- Load podcast transcripts ---
podcast_docs = []
for path in glob.glob("data/podcasts/*.txt"):
    loader = TextLoader(path, encoding="utf-8")
    docs = loader.load()
    episode_name = os.path.basename(path).replace(".txt", "")
    for doc in docs:
        doc.metadata["source_type"] = "podcast"
        doc.metadata["doc_name"] = episode_name
    podcast_docs.extend(docs)

print(f"Loaded {len(podcast_docs)} podcast document(s)")

Loaded 1 podcast document(s)


In [163]:
# --- Chunk with overlap, preserving metadata ---
all_docs = ai_act_docs + podcast_docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
)
chunks = splitter.split_documents(all_docs)

regulation_chunks = [c for c in chunks if c.metadata["source_type"] == "regulation"]
podcast_chunks = [c for c in chunks if c.metadata["source_type"] == "podcast"]

print(f"Total chunks : {len(chunks)}")
print(f"  regulation : {len(regulation_chunks)}")
print(f"  podcast    : {len(podcast_chunks)}")

Total chunks : 311
  regulation : 287
  podcast    : 24


In [164]:
# Sanity-check metadata on a few chunks
for chunk in chunks[:2]:
    print(chunk.metadata)
    print(chunk.page_content[:120])
    print()

{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2019-06-25T08:45:45+02:00', 'moddate': '2019-06-25T08:45:45+02:00', 'source': 'data/eu_ai_act.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1', 'source_type': 'regulation', 'doc_name': 'EU AI Policy'}
GROUPE D’EXPERTS 
INDEPENDANTS DE HAUT NIVEAU SUR 
L’INTELLIGENCE ARTIFICIELLE 
CONSTITUE PAR LA COMMISSION EUROPEENNE E

{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2019-06-25T08:45:45+02:00', 'moddate': '2019-06-25T08:45:45+02:00', 'source': 'data/eu_ai_act.pdf', 'total_pages': 56, 'page': 1, 'page_label': '2', 'source_type': 'regulation', 'doc_name': 'EU AI Policy'}
LIGNES DIRECTRICES EN MATIERE D’ETHIQUE pour UNE IA DIGNE 
DE CONFIANCE 
 
Groupe d’experts de haut niveau sur l’intelli



## Step 2 – Baseline Vector Retrieval

**Objective:** Embed all chunks into a Pinecone index and establish a similarity-search baseline (k=5).  
This is our reference point for measuring the improvements in Steps 3–6.

> **Note:** If the index already exists from a previous run, `from_documents` will upsert into it. To avoid re-indexing costs, skip the `from_documents` call and use `from_existing_index` instead (see commented code below).

In [165]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "trustworthy-ai-rag"

existing_indexes = [i.name for i in pc.list_indexes()]
print("Existing indexes:", existing_indexes)

Existing indexes: ['trustworthy-ai-rag']


In [166]:
# Create index if it doesn't exist yet
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,          # text-embedding-3-small dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Index '{INDEX_NAME}' created")
else:
    print(f"Index '{INDEX_NAME}' already exists – skipping creation")

Index 'trustworthy-ai-rag' already exists – skipping creation


In [167]:
# --- Ingest chunks (skip if already indexed) ---
# Set SKIP_INGEST = True after first run to avoid re-embedding costs
SKIP_INGEST = False

if not SKIP_INGEST:
    print(f"Ingesting {len(chunks)} chunks into Pinecone...")
    vectorstore = PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        index_name=INDEX_NAME,
    )
    print("Ingestion complete")
else:
    vectorstore = PineconeVectorStore.from_existing_index(
        index_name=INDEX_NAME,
        embedding=embeddings,
    )
    print("Connected to existing index")

Ingesting 311 chunks into Pinecone...
Ingestion complete


In [168]:
# Baseline retriever: pure similarity search, top 5
baseline_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

TEST_QUERY = "What are the key principles for trustworthy AI?"

baseline_results = baseline_retriever.invoke(TEST_QUERY)
print(f"Baseline results for: '{TEST_QUERY}'\n")
for i, doc in enumerate(baseline_results, 1):
    print(f"[{i}] source_type={doc.metadata.get('source_type')!r:12s}  doc={doc.metadata.get('doc_name')!r}")
    print(f"     {doc.page_content[:120]}...\n")

Baseline results for: 'What are the key principles for trustworthy AI?'

[1] source_type='regulation'  doc='EU AI Policy'
     valeurs éthiques, ainsi qu’en assurant l’adhésion à ces principes et valeurs, et 3) elle doit être robuste, sur le 
plan...

[2] source_type='regulation'  doc='EU AI Policy'
     valeurs éthiques, ainsi qu’en assurant l’adhésion à ces principes et valeurs, et 3) elle doit être robuste, sur le 
plan...

[3] source_type='regulation'  doc='EU AI Policy'
     valeurs éthiques, ainsi qu’en assurant l’adhésion à ces principes et valeurs, et 3) elle doit être robuste, sur le 
plan...

[4] source_type='regulation'  doc='EU AI Policy'
     valeurs éthiques, ainsi qu’en assurant l’adhésion à ces principes et valeurs, et 3) elle doit être robuste, sur le 
plan...

[5] source_type='regulation'  doc='EU AI Policy'
     valeurs éthiques, ainsi qu’en assurant l’adhésion à ces principes et valeurs, et 3) elle doit être robuste, sur le 
plan...



## Step 3 – LLM-based Relevance Scoring (Advanced)

**Objective:** Use an LLM to score each retrieved chunk 0–1 and re-sort by that score.  
Pattern: retrieve 20 candidates with cheap vector search, then score each with the LLM.

> **Cost note:** 20 sequential LLM calls per query. Use `gpt-4o-mini` to keep this cheap.

In [169]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

scoring_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

scoring_prompt = ChatPromptTemplate.from_template("""
Rate the relevance of the following document to the query on a scale from 0.0 to 1.0.
Output ONLY a single number between 0.0 and 1.0, nothing else.

Query: {query}

Document:
{document}

Relevance score (0.0-1.0):
""")

scoring_chain = scoring_prompt | scoring_llm | StrOutputParser()

def llm_rerank(query: str, docs, top_k: int = 5):
    scored = []
    for doc in docs:
        try:
            raw = scoring_chain.invoke({"query": query, "document": doc.page_content})
            score = float(raw.strip().split()[0])
        except (ValueError, IndexError):
            score = 0.0
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]], [s for s, _ in scored[:top_k]]

print("LLM reranker ready")

LLM reranker ready


In [170]:
# Retrieve 20 candidates, then rerank to top 5
wide_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

candidates = wide_retriever.invoke(TEST_QUERY)
llm_reranked, llm_scores = llm_rerank(TEST_QUERY, candidates, top_k=5)

print(f"LLM-reranked results for: '{TEST_QUERY}'\n")
for i, (doc, score) in enumerate(zip(llm_reranked, llm_scores), 1):
    print(f"[{i}] score={score:.2f}  source_type={doc.metadata.get('source_type')!r:12s}  doc={doc.metadata.get('doc_name')!r}")
    print(f"     {doc.page_content[:120]}...\n")

LLM-reranked results for: 'What are the key principles for trustworthy AI?'

[1] score=1.00  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into clicking somethi...

[2] score=1.00  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into clicking somethi...

[3] score=1.00  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into clicking somethi...

[4] score=1.00  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into clicking somethi...

[5] score=1.00  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into

## Step 4 – Dedicated Reranker (Advanced)

**Objective:** Use a purpose-built reranking model instead of an LLM.  
We implement **both** paths and compare them:

| Path | Model | Cost | Speed |
|------|-------|------|-------|
| A | Cohere Rerank API | Paid per call | Fast |
| B | Cross-encoder `ms-marco-MiniLM-L-6-v2` | Free (local) | Slower (first run downloads ~80 MB) |

> **What's a cross-encoder?** Unlike embeddings (query and doc encoded separately), a cross-encoder reads both query and document *together* and outputs a relevance score directly. Much more accurate but too slow for large corpora – that's why we use it as a reranker on a small candidate set, not as the primary retriever.

In [171]:
# ── Path B: Cross-Encoder (free, no API key needed) ────────────────────────
from sentence_transformers import CrossEncoder

# ~80 MB download on first run – cached locally afterwards
ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def cross_encoder_rerank(query: str, docs, top_k: int = 5):
    pairs = [[query, doc.page_content] for doc in docs]
    scores = ce_model.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in ranked[:top_k]]
    top_scores = [float(s) for s, _ in ranked[:top_k]]
    return top_docs, top_scores

print("Cross-encoder loaded")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2950.75it/s]


Cross-encoder loaded


In [172]:
ce_reranked, ce_scores = cross_encoder_rerank(TEST_QUERY, candidates, top_k=5)

print(f"Cross-encoder reranked results for: '{TEST_QUERY}'\n")
for i, (doc, score) in enumerate(zip(ce_reranked, ce_scores), 1):
    print(f"[{i}] score={score:+.3f}  source_type={doc.metadata.get('source_type')!r:12s}  doc={doc.metadata.get('doc_name')!r}")
    print(f"     {doc.page_content[:120]}...\n")

Cross-encoder reranked results for: 'What are the key principles for trustworthy AI?'

[1] score=+2.682  source_type='podcast'     doc='episode_01'
     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trustworthy AI as having...

[2] score=+2.682  source_type='podcast'     doc='episode_01'
     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trustworthy AI as having...

[3] score=+2.609  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into clicking somethi...

[4] score=+2.609  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It shouldn't deceive you into clicking somethi...

[5] score=+2.609  source_type='podcast'     doc='episode_01'
     the driver's seat. AI shouldn't hurt us or condition us or manipulate us. It should

In [173]:
# ── Path A: Cohere Rerank (requires COHERE_API_KEY) ────────────────────────
#
# We use Path B (cross-encoder) throughout this lab because it is free,
# runs locally, and requires no additional API key. Cohere would offer
# higher throughput and a managed model, but adds a paid dependency.
# Uncomment below if you have a Cohere key and want to compare.

# from langchain_cohere import CohereRerank
# from langchain.retrievers import ContextualCompressionRetriever
#
# cohere_compressor = CohereRerank(
#     model="rerank-english-v3.0",
#     top_n=5,
#     cohere_api_key=os.environ.get("COHERE_API_KEY"),
# )
# cohere_retriever = ContextualCompressionRetriever(
#     base_compressor=cohere_compressor,
#     base_retriever=wide_retriever,
# )
# cohere_results = cohere_retriever.invoke(TEST_QUERY)
# for i, doc in enumerate(cohere_results, 1):
#     print(f"[{i}] {doc.metadata.get('source_type')!r:12s} {doc.page_content[:120]}...")

In [174]:
# ── Path A: Cohere Rerank (requires COHERE_API_KEY) ────────────────────────
# Uncomment to run if you have a Cohere API key

# from langchain_cohere import CohereRerank
# from langchain.retrievers import ContextualCompressionRetriever
#
# cohere_compressor = CohereRerank(
#     model="rerank-english-v3.0",
#     top_n=5,
#     cohere_api_key=os.environ.get("COHERE_API_KEY"),
# )
# cohere_retriever = ContextualCompressionRetriever(
#     base_compressor=cohere_compressor,
#     base_retriever=wide_retriever,
# )
# cohere_results = cohere_retriever.invoke(TEST_QUERY)
# for i, doc in enumerate(cohere_results, 1):
#     print(f"[{i}] {doc.metadata.get('source_type')!r:12s} {doc.page_content[:120]}...")

## Step 5 – Metadata Filtering

**Objective:** Filter at the vector store level *before* similarity calculation so the search only scans the relevant subset.

Without filtering, a question about EU regulations can return podcast tangents with similar embeddings.  
With filtering, we constrain the search space to only regulation chunks (or only podcast chunks).

> **Lesson:** If you skipped metadata in Step 1, you'd have to re-ingest everything here. Plan your metadata schema before indexing.

In [175]:
# Regulation-only retriever
legal_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 10,
        "filter": {"source_type": "regulation"},
    }
)

# Podcast-only retriever
podcast_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 10,
        "filter": {"source_type": "podcast"},
    }
)

In [176]:
legal_query = "What requirements does the regulation set for high-risk AI systems?"

print("=" * 60)
print("UNFILTERED (k=5 baseline)")
print("=" * 60)
unfiltered = baseline_retriever.invoke(legal_query)
for i, doc in enumerate(unfiltered, 1):
    print(f"[{i}] source_type={doc.metadata.get('source_type')!r:12s}  {doc.page_content[:100]}...")

print()
print("=" * 60)
print("FILTERED: regulation only (k=10)")
print("=" * 60)
filtered = legal_retriever.invoke(legal_query)
for i, doc in enumerate(filtered, 1):
    print(f"[{i}] source_type={doc.metadata.get('source_type')!r:12s}  {doc.page_content[:100]}...")

UNFILTERED (k=5 baseline)
[1] source_type='podcast'     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trus...
[2] source_type='podcast'     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trus...
[3] source_type='podcast'     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trus...
[4] source_type='podcast'     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trus...
[5] source_type='podcast'     have tattooed on their arm. Okay, let's start with the big picture. The source material defines trus...

FILTERED: regulation only (k=10)
[1] source_type='regulation'  21 
 
opérateur humain avant de poursuivre son action. 39 Il convient de s’assurer que le système fe...
[2] source_type='regulation'  21 
 
opérateur humain avant de poursuivre son action. 39 Il convient de s’assurer que le système fe...
[3

In [177]:
podcast_query = "How do the podcast guests discuss AI bias and fairness?"

print("=" * 60)
print("UNFILTERED (k=5 baseline)")
print("=" * 60)
unfiltered_pod = baseline_retriever.invoke(podcast_query)
for i, doc in enumerate(unfiltered_pod, 1):
    print(f"[{i}] source_type={doc.metadata.get('source_type')!r:12s}  {doc.page_content[:100]}...")

print()
print("=" * 60)
print("FILTERED: podcast only (k=10)")
print("=" * 60)
filtered_pod = podcast_retriever.invoke(podcast_query)
for i, doc in enumerate(filtered_pod, 1):
    print(f"[{i}] source_type={doc.metadata.get('source_type')!r:12s}  {doc.page_content[:100]}...")

UNFILTERED (k=5 baseline)
[1] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...
[2] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...
[3] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...
[4] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...
[5] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...

FILTERED: podcast only (k=10)
[1] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...
[2] source_type='podcast'     have today. So why are we blowing the dust off this specific report? Because it's not dusty. It's th...
[3] s

## Step 6 – Complete RAG Pipeline with Reranking

**Objective:** Wire metadata filtering + cross-encoder reranking + LCEL into the final production-grade pipeline.

The key insight is that the LCEL chain doesn't care what's inside the retriever – swapping the baseline retriever for the reranking one is a 1-line change.

In [178]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_template("""
You are a legal/policy assistant specialising in AI regulation and AI ethics.
Answer the question using ONLY the provided context.
If the answer is not in the context, say: "I cannot find this information in the provided documents."
Always indicate which document (regulation or podcast) your answer is based on.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('doc_name', 'unknown')} / {d.metadata.get('source_type', '?')}]\n{d.page_content}"
        for d in docs
    )

In [179]:
# --- Helper: wrap cross-encoder reranking as a LangChain Runnable ---

def make_reranking_retriever(base_retriever, top_k=5):
    """Returns a RunnableLambda that retrieves then reranks."""
    def retrieve_and_rerank(query: str):
        candidates = base_retriever.invoke(query)
        if not candidates:
            return []
        reranked, _ = cross_encoder_rerank(query, candidates, top_k=top_k)
        return reranked
    return RunnableLambda(retrieve_and_rerank)


# Wide filtered retriever (k=20) as the candidate source
filtered_wide_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 20, "filter": {"source_type": "regulation"}}
)

final_retriever = make_reranking_retriever(filtered_wide_retriever, top_k=5)

# --- Build the LCEL chain ---
rag_chain = (
    {"context": final_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready")

RAG chain ready


In [180]:
# Quick smoke test
answer = rag_chain.invoke("What are the key principles for trustworthy AI?")
print(answer)

The key principles for trustworthy AI include adherence to ethical values, ensuring robustness both technically and socially to prevent unintentional harm, and the reliability of the AI system as well as the processes and actors involved in its lifecycle. This information is based on the EU AI Policy / regulation.


In [181]:
# --- Baseline chain (for comparison in Step 7) ---
baseline_chain = (
    {"context": baseline_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# --- Filter-only chain – regulation (no reranking) ---
filter_only_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5, "filter": {"source_type": "regulation"}}
)
filter_chain = (
    {"context": filter_only_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# --- Filter-only chain – podcast (no reranking) ---
podcast_filter_chain = (
    {"context": podcast_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# --- Podcast-focused chain with reranking ---
podcast_wide_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 20, "filter": {"source_type": "podcast"}}
)
podcast_rerank_retriever = make_reranking_retriever(podcast_wide_retriever, top_k=5)
podcast_rag_chain = (
    {"context": podcast_rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("All chains ready")

All chains ready


## Step 7 – Performance Evaluation

**Objective:** Compare Baseline vs +Filter vs +Rerank on 7 queries covering both source types.

Evaluation is **manual / qualitative** – for each answer we judge:
1. Correct source type retrieved (regulation vs podcast)?
2. Grounded in documents (no hallucination)?
3. Focuses on substantive passages (not tangential mentions)?
4. Reduces unhelpful "I don't know" answers when content exists?

In [182]:
eval_queries = [
    # Regulation queries
    {"query": "What are the key principles for trustworthy AI?",       "expected_source": "regulation"},
    {"query": "What requirements are set for high-risk AI systems?",    "expected_source": "regulation"},
    {"query": "How does the document define transparency in AI?",       "expected_source": "regulation"},
    {"query": "What human oversight mechanisms are recommended for AI?","expected_source": "regulation"},
    # Podcast queries
    {"query": "How do the speakers discuss AI trust?",                  "expected_source": "podcast"},
    {"query": "What concerns about AI are mentioned in the podcast?",   "expected_source": "podcast"},
    {"query": "What analogies do the speakers use to explain AI?",      "expected_source": "podcast"},
]

print(f"{len(eval_queries)} evaluation queries loaded")

7 evaluation queries loaded


In [183]:
import pathlib

# ── Render comparison table and save to evaluation_table.md ───────────────
TRUNC = 200

manual_judgments = {
    1: ("B",     "All three correct, but Baseline mixes sources (cites podcast). Filter is enough — rerank adds latency without value."),
    2: ("A or B","Rerank REGRESSED to 'cannot find'. Cross-encoder demoted relevant chunks below top-5 cutoff. Baseline & Filter both correct."),
    3: ("A",     "Filter & Rerank BOTH said 'cannot find'. Only Baseline succeeded — relevant chunks scored too low by cross-encoder."),
    4: ("A=B",   "Both detailed; Rerank gives a more abstract answer. Filter is enough."),
    5: ("C",     "Rerank surfaced 'AI worthy of trust + Ethics Guidelines' framing — more substantive than black-box reliance point."),
    6: ("C",     "Rerank found surveillance/tracking concerns; Baseline/Filter only retrieved warfare. Different but more precise."),
    7: ("A or C","Two different valid analogies (bridge vs self-driving car). Subjective which is better — both are in the transcript."),
}

lines = [
    "| # | Query | Expected | Baseline (A) | + Filter (B) | + Rerank (C) | Best | Why |",
    "|---|-------|----------|--------------|--------------|--------------|------|-----|",
]
for i, r in enumerate(eval_results, 1):
    a = r["baseline"][:TRUNC].replace("\n", " ").replace("|", "/")
    b = r["filter"][:TRUNC].replace("\n", " ").replace("|", "/")
    c = r["rerank"][:TRUNC].replace("\n", " ").replace("|", "/")
    best, why = manual_judgments.get(i, ("?", ""))
    lines.append(f"| {i} | {r['query']} | {r['expected_source']} | {a} | {b} | {c} | {best} | {why} |")

table = "\n".join(lines)
pathlib.Path("evaluation_table.md").write_text(table, encoding="utf-8")
print("Saved to evaluation_table.md")
print()
print(table)

Saved to evaluation_table.md

| # | Query | Expected | Baseline (A) | + Filter (B) | + Rerank (C) | Best | Why |
|---|-------|----------|--------------|--------------|--------------|------|-----|
| 1 | What are the key principles for trustworthy AI? | regulation | The key principles for trustworthy AI, as outlined in the podcast, are:   1. Lawfulness - The AI must comply with all regulations. 2. Prevention of harm - AI should not hurt, condition, manipulate, or | The key principles for trustworthy AI are:   1. It must be lawful, ensuring compliance with applicable laws and regulations;   2. It must be ethical, ensuring adherence to ethical principles and value | The key principles for trustworthy AI are: a) it must be lawful, ensuring compliance with applicable laws and regulations; b) it must be ethical, adhering to ethical principles and values; and c) it m | B | All three correct, but Baseline mixes sources (cites podcast). Filter is enough — rerank adds latency without value. |
| 2

In [184]:
# ── Render a compact comparison table ─────────────────────────────────────
# Truncate answers to 200 chars for readability
TRUNC = 200

header = "| # | Query | Expected Source | Baseline (A) | + Filter (B) | + Rerank (C) | Best |\n"
header += "|---|-------|-----------------|--------------|--------------|--------------|------|"
print(header)

for i, r in enumerate(eval_results, 1):
    a = r["baseline"][:TRUNC].replace("\n", " ")
    b = r["filter"][:TRUNC].replace("\n", " ")
    c = r["rerank"][:TRUNC].replace("\n", " ")
    print(f"| {i} | {r['query'][:60]} | {r['expected_source']} | {a} | {b} | {c} | ??? |")

| # | Query | Expected Source | Baseline (A) | + Filter (B) | + Rerank (C) | Best |
|---|-------|-----------------|--------------|--------------|--------------|------|
| 1 | What are the key principles for trustworthy AI? | regulation | The key principles for trustworthy AI, as outlined in the podcast, are:   1. Lawfulness - The AI must comply with all regulations. 2. Prevention of harm - AI should not hurt, condition, manipulate, or | The key principles for trustworthy AI are:   1. It must be lawful, ensuring compliance with applicable laws and regulations;   2. It must be ethical, ensuring adherence to ethical principles and value | The key principles for trustworthy AI are: a) it must be lawful, ensuring compliance with applicable laws and regulations; b) it must be ethical, adhering to ethical principles and values; and c) it m | ??? |
| 2 | What requirements are set for high-risk AI systems? | regulation | The requirements for high-risk AI systems include ensuring that the system 

In [185]:
# ── "I cannot find this information" rate ─────────────────────────────────
# Measures how often each pipeline fails to answer — lower is better.

def is_unhelpful(answer: str) -> bool:
    return "cannot find" in answer.lower() or "not in the context" in answer.lower()

rate_a = sum(is_unhelpful(r["baseline"]) for r in eval_results)
rate_b = sum(is_unhelpful(r["filter"])   for r in eval_results)
rate_c = sum(is_unhelpful(r["rerank"])   for r in eval_results)
n = len(eval_results)

print("'I cannot find this information' rate (lower is better):")
print(f"  [A] Baseline       : {rate_a}/{n}")
print(f"  [B] + Filter       : {rate_b}/{n}")
print(f"  [C] + Filter+Rerank: {rate_c}/{n}")
if rate_c > rate_a:
    print(f"\n⚠️  Rerank doubled the unhelpful-answer rate vs Baseline ({rate_c} vs {rate_a}).")
    print("   Cross-encoder precision cuts both ways: it removes noise AND occasionally demotes relevant chunks.")

'I cannot find this information' rate (lower is better):
  [A] Baseline       : 0/7
  [B] + Filter       : 1/7
  [C] + Filter+Rerank: 2/7

⚠️  Rerank doubled the unhelpful-answer rate vs Baseline (2 vs 0).
   Cross-encoder precision cuts both ways: it removes noise AND occasionally demotes relevant chunks.


### Hypothesis test: was the regression model-specific or approach-specific?

Q2 and Q3 regressed under `ms-marco-MiniLM-L-6-v2`. Two possible explanations:
1. **Model bottleneck** — MiniLM is too small to score legal text accurately; a stronger model would fix it.
2. **Approach bottleneck** — reranking on 800-char chunks of structured legal text is fundamentally problematic regardless of model.

We test by running `ms-marco-electra-base` (~400 MB, larger model) on the same candidates for Q2 and Q3.  
- If ELECTRA retrieves the right chunk → the regression was model-specific.  
- If ELECTRA also fails → chunking or query formulation is the real issue.

In [186]:
# ~400 MB download on first run – significantly larger than MiniLM
ce_strong = CrossEncoder("cross-encoder/ms-marco-electra-base")

def cross_encoder_rerank_strong(query, docs, top_k=5):
    pairs = [[query, doc.page_content] for doc in docs]
    scores = ce_strong.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

# The two queries that regressed under MiniLM
problem_queries = [
    "What requirements are set for high-risk AI systems?",   # Q2
    "How does the document define transparency in AI?",      # Q3
]

for q in problem_queries:
    candidates_q = wide_retriever.invoke(q)
    minilm_top, _ = cross_encoder_rerank(q, candidates_q, top_k=3)
    electra_top   = cross_encoder_rerank_strong(q, candidates_q, top_k=3)

    print(f"Query: {q}")
    print(f"  MiniLM  top-1: {minilm_top[0].page_content[:120]}")
    print(f"  ELECTRA top-1: {electra_top[0].page_content[:120]}")
    print()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1051.47it/s]


Query: What requirements are set for high-risk AI systems?
  MiniLM  top-1: valeurs éthiques, ainsi qu’en assurant l’adhésion à ces principes et valeurs, et 3) elle doit être robuste, sur le 
plan
  ELECTRA top-1: peut être modifié, ce qui conduit le système à prendre des décisions différentes voire à s’arrêter. Les systèmes 
et les

Query: How does the document define transparency in AI?
  MiniLM  top-1: Yeah. You might not tell an algorithm your political views or your sexual orientation, but based on your location data, 
  ELECTRA top-1: Yeah. You might not tell an algorithm your political views or your sexual orientation, but based on your location data, 



In [187]:
# ── Latency summary ────────────────────────────────────────────────────────
import statistics

avg_a = statistics.mean(r["time_baseline"] for r in eval_results)
avg_b = statistics.mean(r["time_filter"] for r in eval_results)
avg_c = statistics.mean(r["time_rerank"] for r in eval_results)

print(f"Average latency per query:")
print(f"  [A] Baseline          : {avg_a:.1f}s")
print(f"  [B] + Filter          : {avg_b:.1f}s")
print(f"  [C] + Filter + Rerank : {avg_c:.1f}s")
print(f"Reranking overhead      : +{avg_c - avg_b:.1f}s per query")

Average latency per query:
  [A] Baseline          : 4.0s
  [B] + Filter          : 3.7s
  [C] + Filter + Rerank : 12.8s
Reranking overhead      : +9.1s per query


## Summary & Key Takeaways

Fill in the `Best` column of the table above after reviewing the outputs manually, then write `lab_summary.md` based on what you observe.

**Reflection questions:**
- Which query showed the biggest jump from Baseline to Reranked?
- Which query was already perfect at Baseline? What does that tell you?
- Did filtering matter more or less than reranking for your query set?
- At what latency cost does reranking become unjustifiable?
- How did mixing regulation and podcast chunks affect Baseline answers?